In [3]:
from itertools import cycle
import numpy as np
import matplotlib.pyplot as plt
import scipy
import pandas as pd
import xgboost as xgb
from sklearn import linear_model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler 
from sklearn.datasets import make_regression
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_curve, auc
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import median_absolute_error
from sklearn.metrics import max_error
from string import ascii_uppercase
from geopy.distance import geodesic

In [4]:
from src.cleaning_and_helpers import plot_test_preds

In [5]:
np.random.seed(1298)

Import X and y from data prep steps

In [6]:
SI_X = np.load('../pollenGeolocation_saved/SI_X.npy')
SI_y = np.load('../pollenGeolocation_saved/SI_y.npy')

FFAR_X = np.load('../pollenGeolocation_saved/FFAR_X.npy')
FFAR_y = np.load('../pollenGeolocation_saved/FFAR_y.npy')

NCASI_X = np.load('../pollenGeolocation_saved/NCASI_X.npy')
NCASI_y = np.load('../pollenGeolocation_saved/NCASI_y.npy')

In [7]:
# Train test split for each project
seed = 1298
test_size = 0.20

# --- Function to split each project ---
def split_project(X, y, test_size, random_state):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

# --- Split each project ---
SI_X_train, SI_X_test, SI_y_train, SI_y_test = split_project(SI_X, SI_y, test_size, seed)
NCASI_X_train, NCASI_X_test, NCASI_y_train, NCASI_y_test = split_project(NCASI_X, NCASI_y, test_size, seed)
FFAR_X_train, FFAR_X_test, FFAR_y_train, FFAR_y_test = split_project(FFAR_X, FFAR_y, test_size, seed)

# --- Concatenate train and test splits from all projects ---
X_train = np.concatenate([SI_X_train, NCASI_X_train, FFAR_X_train], axis=0)
y_train = np.concatenate([SI_y_train, NCASI_y_train, FFAR_y_train], axis=0)

X_test = np.concatenate([SI_X_test, NCASI_X_test, FFAR_X_test], axis=0)
y_test = np.concatenate([SI_y_test, NCASI_y_test, FFAR_y_test], axis=0)

# --- Standardize X and y using training data only ---
sc_X = StandardScaler()
sc_y = StandardScaler()

X_train = sc_X.fit_transform(X_train)
y_train = sc_y.fit_transform(y_train)

X_test = sc_X.transform(X_test)
y_test = sc_y.transform(y_test)


In [ ]:
from sklearn.model_selection import GridSearchCV, RepeatedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import MultiTaskLasso
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import make_scorer
import numpy as np

# -----------------------------
# Custom RMSE scorer
# -----------------------------
def multioutput_rmse(y_true, y_pred):
    return np.sqrt(((y_true - y_pred) ** 2).mean())

rmse_scorer = make_scorer(multioutput_rmse, greater_is_better=False)

# -----------------------------
# Cross-validation setup
# -----------------------------
cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=1298)

# -----------------------------
# Models and param grids
# -----------------------------
models = {
    "MultiTaskLasso": (
        linear_model.MultiTaskLasso(max_iter=10000),
        {
            'estimator__alpha': [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]
        }
    ),

    "SVR": (
        MultiOutputRegressor(SVR()),
        {
            'estimator__estimator__C': [0.1, 1, 10],
            'estimator__estimator__gamma': ['scale', 'auto']
        }  # 3 × 2 = 6
    ),

    "KNN": (
        MultiOutputRegressor(KNeighborsRegressor()),
        {
            'estimator__estimator__n_neighbors': [1, 3, 5, 7, 9, 11]
        }
    ),

    "DecisionTree": (
        MultiOutputRegressor(DecisionTreeRegressor()),
        {
            'estimator__estimator__max_depth': [3, 5, 10, 15, 20, None]
        }
    ),

    "RandomForest": (
        MultiOutputRegressor(RandomForestRegressor()),
        {
            'estimator__estimator__n_estimators': [50, 100, 150],
            'estimator__estimator__max_depth': [10, 20]
        }  # 3 × 2 = 6
    ),

    "XGBoost": (
        MultiOutputRegressor(XGBRegressor(objective='reg:squarederror')),
        {
            'estimator__estimator__n_estimators': [100, 200],
            'estimator__estimator__max_depth': [3, 6, 9]
        }  # 2 × 3 = 6
    )
}



# -----------------------------
# Run GridSearchCV for all models
# -----------------------------
results = {}

for name, (model, param_grid) in models.items():
    print(f"Training model: {name}")
    
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('estimator', model)
    ])
    
    search = GridSearchCV(
        pipeline,
        param_grid=param_grid,
        scoring=rmse_scorer,
        cv=cv,
        n_jobs=-1,
        verbose=1
    )
    
    search.fit(X_train, y_train)
    results[name] = search

#--------------------------
# Compare Results
#--------------------------
for name, search in results.items():
    print(f"{name}: Best CV RMSE = {-search.best_score_:.3f}")

Training model: MultiTaskLasso
Fitting 15 folds for each of 6 candidates, totalling 90 fits
Training model: SVR
Fitting 15 folds for each of 6 candidates, totalling 90 fits
Training model: KNN
Fitting 15 folds for each of 6 candidates, totalling 90 fits
Training model: DecisionTree
Fitting 15 folds for each of 6 candidates, totalling 90 fits
Training model: RandomForest
Fitting 15 folds for each of 6 candidates, totalling 90 fits
Training model: XGBoost
Fitting 15 folds for each of 6 candidates, totalling 90 fits
MultiTaskLasso: Best CV RMSE = 0.830
SVR: Best CV RMSE = 0.712
KNN: Best CV RMSE = 0.456
DecisionTree: Best CV RMSE = 0.517
RandomForest: Best CV RMSE = 0.570
XGBoost: Best CV RMSE = 0.469


In [9]:
for name, search in results.items():
    print(f"{name} best params:")
    print(search.best_params_)
    print()


MultiTaskLasso best params:
{'estimator__alpha': 0.1}

SVR best params:
{'estimator__estimator__C': 10, 'estimator__estimator__gamma': 'auto'}

KNN best params:
{'estimator__estimator__n_neighbors': 1}

DecisionTree best params:
{'estimator__estimator__max_depth': None}

RandomForest best params:
{'estimator__estimator__max_depth': 20, 'estimator__estimator__n_estimators': 150}

XGBoost best params:
{'estimator__estimator__max_depth': 9, 'estimator__estimator__n_estimators': 200}



In [11]:
param_grid

{'estimator__estimator__n_estimators': [100, 200],
 'estimator__estimator__max_depth': [3, 6, 9]}

In [8]:
from sklearn.metrics import r2_score, mean_squared_error, median_absolute_error
from geopy.distance import geodesic
import numpy as np

def evaluate_model(name, model_class, best_params, X_train, y_train, X_test, y_test):
    """
    Initialize, fit, predict, and evaluate a model using best hyperparameters.

    Parameters:
    - name: string, name of model
    - model_class: class or callable that returns a model
    - best_params: dict of hyperparameters
    - X_train, y_train, X_test, y_test: data splits

    Returns:
    - dict with performance metrics and predictions
    """
    print(f"Evaluating {name}...")

    # If model_class is a callable (e.g., lambda), call it to get the estimator
    model = model_class(**best_params)

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    r2 = r2_score(y_test, preds)
    rmse = mean_squared_error(y_test, preds, squared=False)
    mae = median_absolute_error(y_test, preds)
    distances = [geodesic(real, pred).kilometers for real, pred in zip(y_test, preds)]
    avg_dist = np.mean(distances)
    se_dist = np.std(distances)

    return {
        "model": name,
        "r2": r2,
        "rmse": rmse,
        "mae": mae,
        "avg_km_error": avg_dist,
        "se_km_error": se_dist,
        "preds": preds
    }


In [ ]:
from sklearn.linear_model import MultiTaskLasso
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor

# Define model mapping (must match what was passed into GridSearchCV)
model_classes = {
    "MultiTaskLasso": MultiTaskLasso,
    "SVR": lambda **kwargs: MultiOutputRegressor(SVR(**kwargs)),
    "KNN": lambda **kwargs: MultiOutputRegressor(KNeighborsRegressor(**kwargs)),
    "DecisionTree": lambda **kwargs: MultiOutputRegressor(DecisionTreeRegressor( **kwargs)),
    "RandomForest": lambda **kwargs: MultiOutputRegressor(RandomForestRegressor( **kwargs)),
    "XGBoost": lambda **kwargs: MultiOutputRegressor(XGBRegressor(objective="reg:squarederror", **kwargs))
}

# Collect results
all_metrics = []

for name, search in results.items():
    best_params = search.best_params_
    
    # Flatten parameter dict for instantiation (remove 'estimator__estimator__' prefix)
    flat_params = {
        k.replace("estimator__estimator__", "").replace("estimator__", ""): v
        for k, v in best_params.items()
    }

    metrics = evaluate_model(
        name=name,
        model_class=model_classes[name],
        best_params=flat_params,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test
    )
    all_metrics.append(metrics)


Evaluating MultiTaskLasso...
Evaluating SVR...
Evaluating KNN...
Evaluating DecisionTree...
Evaluating RandomForest...
Evaluating XGBoost...


In [10]:
import pandas as pd

results_df = pd.DataFrame([{
    "Model": m["model"],
    "R²": m["r2"],
    "RMSE": m["rmse"],
    "MAE": m["mae"],
    "Avg Geodesic Error (km)": m["avg_km_error"],
    "SE Geodesic Error (km)": m["se_km_error"]
} for m in all_metrics])

results_df
#results_df.to_csv("model_comparison_results.csv", index=False)


,Model,R²,RMSE,MAE,Avg Geodesic Error (km),SE Geodesic Error (km)
0,MultiTaskLasso,0.347065,0.811904,0.268774,86.284926,94.123849
1,SVR,0.612362,0.629532,0.094468,58.631424,79.482927
2,KNN,0.870079,0.363229,0.005584,17.148513,54.405363
3,DecisionTree,0.777986,0.476525,0.011114,22.146429,71.404898
4,RandomForest,0.757216,0.499169,0.028488,42.060480,66.120808
5,XGBoost,0.806542,0.445505,0.054433,35.483505,60.251008


In [ ]:
## anything below is old

In [ ]:
# # Extract rbcl columns
# SI_cols = [col for col in df_si if col.startswith('RBCL')]
# SI_df = df_si[SI_cols]

# FFAR_cols = [col for col in df_ffar if col.startswith('RBCL')]
# FFAR_df = df_ffar[FFAR_cols]

# NCASI_cols = [col for col in df_ncasi if col.startswith('RBCL')]
# NCASI_df = df_ncasi[NCASI_cols]

# # Get union of all features
# all_features = sorted(set(SI_df.columns).union(NCASI_df.columns).union(FFAR_df.columns))

# # Reindex all with full set, filling missing with 0
# SI_df = SI_df.reindex(columns=all_features, fill_value=0)
# NCASI_df = NCASI_df.reindex(columns=all_features, fill_value=0)
# FFAR_df = FFAR_df.reindex(columns=all_features, fill_value=0)

# # Convert to numpy
# SI_X = SI_df.to_numpy()
# NCASI_X = NCASI_df.to_numpy()
# FFAR_X = FFAR_df.to_numpy()


In [ ]:
# Train test split for each project
seed = 1298
test_size = 0.20

# --- Function to split each project ---
def split_project(X, y, test_size, random_state):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

# --- Split each project ---
SI_X_train, SI_X_test, SI_y_train, SI_y_test = split_project(SI_X, SI_y, test_size, seed)
NCASI_X_train, NCASI_X_test, NCASI_y_train, NCASI_y_test = split_project(NCASI_X, NCASI_y, test_size, seed)
FFAR_X_train, FFAR_X_test, FFAR_y_train, FFAR_y_test = split_project(FFAR_X, FFAR_y, test_size, seed)

# --- Concatenate train and test splits from all projects ---
X_train = np.concatenate([SI_X_train, NCASI_X_train, FFAR_X_train], axis=0)
y_train = np.concatenate([SI_y_train, NCASI_y_train, FFAR_y_train], axis=0)

X_test = np.concatenate([SI_X_test, NCASI_X_test, FFAR_X_test], axis=0)
y_test = np.concatenate([SI_y_test, NCASI_y_test, FFAR_y_test], axis=0)

# --- Standardize X and y using training data only ---
sc_X = StandardScaler()
sc_y = StandardScaler()

X_train = sc_X.fit_transform(X_train)
y_train = sc_y.fit_transform(y_train)

X_test = sc_X.transform(X_test)
y_test = sc_y.transform(y_test)


In [ ]:
# # Train/test split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=1298)

# # Create scaler for X and y
# sc_X = StandardScaler()
# sc_y = StandardScaler()

# # Scale training data
# X_train = sc_X.fit_transform(X_train)
# y_train = sc_y.fit_transform(y_train)  

# # Scale test data using training parameters
# X_test = sc_X.transform(X_test)
# y_test = sc_y.transform(y_test)  

# Linear regression  
https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.MultiTaskLasso.html

In [ ]:
# Create a figure and an axis
#fig, ax = plt.subplots(figsize=(8, 6))


MTKNReg = linear_model.MultiTaskLasso(alpha=0.05)
MTKNReg.fit(X_train, y_train)
preds_LR = MTKNReg.predict(X_test)

LR_rsq = r2_score(y_test,preds_LR)
LR_mse = mean_squared_error(y_test,preds_LR)
LR_mae = median_absolute_error(y_test,preds_LR)
#LR_maxerror = max_error(y_test,preds_LR)
distances_LR = [geodesic(real, pred).kilometers for real, pred in zip(y_test, preds_LR)]
avg_distance_LR = np.mean(distances_LR)
se_distance_LR = np.std(distances_LR) 


#plot_test_preds(y_test, preds_LR, sc_y, "Linear Regression",ax=ax)

In [ ]:
from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error,r2_score,mean_squared_error

# SVR Support Vector Regression multioutput regressor
https://medium.com/pursuitnotes/support-vector-regression-in-6-steps-with-python-c4569acd062d  
https://scikit-learn.org/stable/modules/svm.html#regression
https://datascience.stackexchange.com/questions/82743/valueerror-y-should-be-a-1d-array-got-an-array-of-shape-285-30-instead  
https://www.section.io/engineering-education/support-vector-regression-in-python/

In [ ]:
# Create a figure and an axis
#fig, ax = plt.subplots(figsize=(8, 6))

# Create the SVR regressor
svr = SVR(epsilon=0.01)

# Create the Multioutput Regressor
mor = MultiOutputRegressor(svr)

# Train the regressor
mor = mor.fit(X_train, y_train)

# Generate predictions for testing data
preds_SVR = mor.predict(X_test)

SVR_rsq = r2_score(y_test,preds_SVR)
SVR_mse = mean_squared_error(y_test,preds_SVR)
SVR_mae = median_absolute_error(y_test,preds_SVR)
#SVR_maxerror = max_error(y_test,preds_SVR)
distances_SVR = [geodesic(real, pred).kilometers for real, pred in zip(y_test, preds_SVR)]
avg_distance_SVR = np.mean(distances_SVR)
se_distance_SVR = np.std(distances_SVR) 

#plot_test_preds(y_test, preds_SVR, sc_y, "Support Vector Regression", ax=ax)

# k-Nearest Neighbors for Multioutput Regression
https://machinelearningmastery.com/multi-output-regression-models-with-python/

In [ ]:
# Create a figure and an axis
#fig, ax = plt.subplots(figsize=(8, 6))

# fit model
KNR = KNeighborsRegressor(algorithm='auto', weights='distance', p=2, n_neighbors=2) 
KNR = KNR.fit(X_train, y_train)

preds_KNR = KNR.predict(X_test)

KNR_rsq = r2_score(y_test,preds_KNR)
KNR_mse = mean_squared_error(y_test,preds_KNR)
KNR_mae = median_absolute_error(y_test,preds_KNR)
#KNR_maxerror = max_error(y_test,preds_KNR)
distances_KNR = [geodesic(real, pred).kilometers for real, pred in zip(y_test, preds_KNR)]
avg_distance_KNR = np.mean(distances_KNR)
se_distance_KNR = np.std(distances_KNR) 

#plot_test_preds(y_test, preds_KNR, sc_y, "k-Nearest Neighbors", ax=ax)

# Decision Tree for multioutput regression  


In [ ]:
# Create a figure and an axis
#fig, ax = plt.subplots(figsize=(8, 6))

# fit model
DTR = DecisionTreeRegressor(max_depth=17, min_samples_split=2, min_samples_leaf=1, max_leaf_nodes=70, max_features='auto')
DTR = DTR.fit(X_train, y_train)

preds_DTR = DTR.predict(X_test)

DTR_rsq = r2_score(y_test,preds_DTR)
DTR_mse = mean_squared_error(y_test,preds_DTR)
DTR_mae = median_absolute_error(y_test,preds_DTR)
#DTR_maxerror = max_error(y_test,preds_DTR)
distances_DTR = [geodesic(real, pred).kilometers for real, pred in zip(y_test, preds_DTR)]
avg_distance_DTR = np.mean(distances_DTR)
se_distance_DTR = np.std(distances_DTR) 


#plot_test_preds(y_test, preds_DTR, sc_y, "Decision Tree", ax=ax)

In [ ]:
#Random forest
# Create a figure and an axis
#fig, ax = plt.subplots(figsize=(8, 6))

# Using tuned model
rf =RandomForestRegressor(n_estimators = 1000,
                          min_samples_split = 20, 
                          min_samples_leaf = 1, 
                          #max_samples=0.75,
                          max_features = 'auto', 
                          max_depth = 20, 
                          bootstrap = False) 
rf = rf.fit(X_train,y_train)

pred_train = rf.predict(X_train)
preds_RF = rf.predict(X_test)

RF_rsq = r2_score(y_test,preds_RF)
RF_mse = mean_squared_error(y_test,preds_RF)
RF_mae = median_absolute_error(y_test,preds_RF)
#RF_maxerror = max_error(y_test,preds_RF)
distances_RF = [geodesic(real, pred).kilometers for real, pred in zip(y_test, preds_RF)]
avg_distance_RF = np.mean(distances_RF)
se_distance_RF = np.std(distances_RF) 

#plot_test_preds(y_test, preds_RF, sc_y, "Random Forest", ax=ax)

In [ ]:
""" # Select the top 25 features
top_features = feat_importances.nlargest(25).sort_values()

# Create a figure and axis
fig, ax = plt.subplots(figsize=(8, 6))

# Set Seaborn style for better aesthetics
sns.set_style("ticks")

# Create horizontal bar plot with thicker bars
bars = ax.barh(
    top_features.index,
    top_features.values,
    color=sns.color_palette("magma", len(top_features)),
    edgecolor='black',
    height=0.8  # Thicker bars
)

# Adjust x-axis to add padding for annotations
x_max = top_features.max()  # Find the maximum importance
ax.set_xlim(0, x_max + 0.03)  # Add 5% padding to the right

# Add axis labels
ax.set_xlabel("Feature Importance", fontsize=12)
ax.set_ylabel("Pollen ID", fontsize=12)

# Add annotations at the end of each bar
for bar in bars:
    ax.text(
        bar.get_width() + 0.005,  # Slightly past the bar's end
        bar.get_y() + bar.get_height() / 2,  # Center vertically on the bar
        f"{bar.get_width():.3f}",  # Format the number
        va='center', fontsize=10, color='black'
    )

# Tidy layout and display plot
fig.tight_layout()
plt.show() """

https://towardsdatascience.com/hyperparameter-tuning-the-random-forest-in-python-using-scikit-learn-28d2aa77dd74

In [ ]:
RF_tuning = False
if RF_tuning == True:
    from sklearn.model_selection import RandomizedSearchCV
    # Number of trees in random forest
    n_estimators = [int(x) for x in np.linspace(start = 200, stop = 2000, num = 10)]
    # Number of features to consider at every split
    max_features = ['auto', 'sqrt']
    # Maximum number of levels in tree
    max_depth = [int(x) for x in np.linspace(10, 110, num = 11)]
    max_depth.append(None)
    # Minimum number of samples required to split a node
    min_samples_split = [2, 5, 10]
    # Minimum number of samples required at each leaf node
    min_samples_leaf = [1, 2, 4]
    # Method of selecting samples for training each tree
    bootstrap = [True, False]
    # Create the random grid
    random_grid = {'n_estimators': n_estimators,
                'max_features': max_features,
                'max_depth': max_depth,
                'min_samples_split': min_samples_split,
                'min_samples_leaf': min_samples_leaf,
                'bootstrap': bootstrap}
    print(random_grid)

    rf_random = RandomizedSearchCV(estimator = rf, param_distributions = random_grid, n_iter = 100, cv = 3, verbose=2, random_state=42, n_jobs = -1)
    # Fit the random search model
    rf_random.fit(X_train, y_train)

    rf_random.best_params_



XGBoost

In [ ]:
from xgboost import XGBRegressor

# Create a figure and an axis
#fig, ax = plt.subplots(figsize=(8, 6))

# Create the base XGBoost regressor
xgb = XGBRegressor(objective='reg:squarederror', 
                   reg_lambda=10,
                   subsample=0.6,
                   reg_alpha=0,
                   n_estimators=500,
                   min_child_weight=1,
                   max_depth=15,
                   learning_rate=0.2,
                   gamma=0.2,
                   colsample_bytree=1.0
                   )

# Wrap with MultiOutputRegressor for multioutput regression
multi_xgb = MultiOutputRegressor(xgb)

# Train the model
multi_xgb.fit(X_train, y_train)

# Generate predictions
preds_xg = multi_xgb.predict(X_test)

XG_rsq = r2_score(y_test,preds_xg)
XG_mse = mean_squared_error(y_test,preds_xg)

#plot_test_preds(y_test, preds_xg, sc_y, "XGBoost", ax=ax)


In [ ]:
tune_XG = True

if tune_XG == True:
    # hyperparam tuning for xgboost
    # Example hyperparameter grid for XGBRegressor
    param_grid = {
        'estimator__n_estimators': [50, 100, 200],
        'estimator__max_depth': [3, 5, 7],
        'estimator__learning_rate': [0.01, 0.1, 0.2],
        'estimator__subsample': [0.8, 1.0],
        'estimator__colsample_bytree': [0.8, 1.0],
        'estimator__reg_alpha': [0, 0.1, 1],  # L1 regularization
        'estimator__reg_lambda': [0.1, 1, 10],  # L2 regularization
    }
    # Set up RandomizedSearchCV
    xgb_random = RandomizedSearchCV(
        estimator=multi_xgb,
        param_distributions=param_grid,
        n_iter=100,  # Number of parameter settings sampled
        cv=3,        # 3-fold cross-validation
        verbose=2,
        random_state=42,
        n_jobs=-1    # Use all available cores
    )
    # Fit the RandomizedSearchCV
    xgb_random.fit(X_train, y_train)

    # Get the best parameters
    print("Best parameters found: ", xgb_random.best_params_)


In [ ]:
# TODO save out best params so don't have to rerun cross validation eatch time

In [ ]:
# Create a figure and an axis
#fig, ax = plt.subplots(figsize=(8, 6))

# Get predictions using the best estimator
best_xgb = xgb_random.best_estimator_
preds_XG_best = best_xgb.predict(X_test)

XG_rsq_best = r2_score(y_test,preds_XG_best)
XG_mse_best = mean_squared_error(y_test,preds_XG_best)
XG_mae_best = median_absolute_error(y_test,preds_XG_best)
#XG_maxerror_best = max_error(y_test,preds_XG_best)
distances_XG = [geodesic(real, pred).kilometers for real, pred in zip(y_test, preds_XG_best)]
avg_distance_XG = np.mean(distances_XG)
se_distance_XG = np.std(distances_XG) 

#plot_test_preds(y_test, preds_XG_best, sc_y, "XGBoost", ax=ax)


Multilayer perceptron

In [ ]:
""" from tensorflow.keras.models import load_model

# Load the saved model
mlp_model = load_model("../pollenGeolocation_saved/best_CNN_model.h5")

# Make predictions
preds_MLP = mlp_model.predict(X_test)

MLP_rsq = r2_score(y_test,preds_MLP)
MLP_mse = mean_squared_error(y_test,preds_MLP)
distances_MLP = [geodesic(real, pred).kilometers for real, pred in zip(y_test, preds_MLP)]
avg_distance_MLP = np.mean(distances_MLP)
se_distance_MLP = np.std(distances_MLP) 

# Create a figure and an axis
fig, ax = plt.subplots(figsize=(8, 6))

plot_test_preds(y_test, preds_MLP, sc_y, "MLP",ax=ax) """

CNN

In [ ]:
""" from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Flatten, Dense

CNN = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(182, 1)),
    Conv1D(filters=32, kernel_size=3, activation='relu'),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(2)  # Output layer
])

CNN.compile(optimizer='adam', loss='mse', metrics=['mae'])

CNN.fit(X_train, y_train, validation_split=0.2, epochs=200)

# Make predictions
preds_CNN = CNN.predict(X_test)

CNN_rsq = r2_score(y_test,preds_CNN)
CNN_mse = mean_squared_error(y_test,preds_CNN)
distances_CNN = [geodesic(real, pred).kilometers for real, pred in zip(y_test, preds_CNN)]
avg_distance_CNN = np.mean(distances_CNN)
se_distance_CNN = np.std(distances_CNN) 

# Create a figure and an axis
fig, ax = plt.subplots(figsize=(8, 6))

plot_test_preds(y_test, preds_CNN, sc_y, "CNN",ax=ax)


 """

In [ ]:
## TODO drop outliers, add NN results to tables and visualizations below

In [ ]:
method = ['Linear Regression', 
          'SVR' , 
          'Decision Tree', 
          'k-Nearest Neighbors', 
          'XGBoost', 
          'Random Forest'
          #'Multilayer Perceptron',
          #'Convolutional Neural Network'
          ]
rsquared = [LR_rsq, 
            SVR_rsq, 
            DTR_rsq,
            KNR_rsq, 
            XG_rsq_best,
            RF_rsq
            #MLP_rsq,
           # CNN_rsq
           ]
MSE = [LR_mse, 
       SVR_mse, 
       DTR_mse,
       KNR_mse, 
       XG_mse_best,
       RF_mse
       #MLP_mse,
       #CNN_mse
       ]
dist = [avg_distance_LR,
        avg_distance_SVR,
        avg_distance_DTR,
        avg_distance_KNR,
        avg_distance_XG,
        avg_distance_RF
        #avg_distance_MLP,
        #avg_distance_CNN
        ]

se = [se_distance_LR,
      se_distance_SVR,
      se_distance_DTR,
      se_distance_KNR,
      se_distance_XG,
      se_distance_RF
      #se_distance_MLP,
      #se_distance_CNN
      ]

MAE = [LR_mae,
       SVR_mae,
       DTR_mae,
       KNR_mae,
       XG_mae_best,
       RF_mae]


df = pd.DataFrame({'Model' : method,
              'r2' : rsquared,
              'MSE' : MSE,
              'MAE' : MAE,
              'AvgDistLoss' : dist
              })
df

df.to_csv("tables/model_results.csv")

In [ ]:
import seaborn as sns
df

In [ ]:
# Generate LaTeX table
latex_table = df.to_latex(index=False)
print(latex_table)

In [ ]:

# Combine data into a DataFrame
data = {
    "model": ["Linear Regression"] * len(distances_LR) +
             ["SVR"] * len(distances_SVR) +
             ["Decision Tree"] * len(distances_DTR) +
             ["k-NN"] * len(distances_KNR) +
             ["XGBoost"] * len(distances_XG) +
             ["Random Forest"] * len(distances_RF),
             #["CNN"] * len(distances_CNN),
    "distance_loss": distances_LR + distances_SVR + distances_DTR + distances_KNR + distances_XG +
                     distances_RF, #+ distances_CNN,
    "distance_std": se_distance_LR + se_distance_SVR + se_distance_DTR + se_distance_KNR + se_distance_XG + se_distance_RF #+ se_distance_CNN
}

df = pd.DataFrame(data)


# # Combine data into a DataFrame
# data = {
#     "model": ["Linear Regression"] * len(distances_LR) +
#              ["SVR"] * len(distances_SVR) +
#              ["Decision Tree"] * len(distances_DTR) +
#              ["k-NN"] * len(distances_KNR) +
#              ["XGBoost"] * len(distances_XG) +
#              ["Random Forest"] * len(distances_RF),
#     "distance_loss": distances_LR + distances_SVR + distances_DTR + distances_KNR + distances_XG +
#                      distances_RF,
# }

# df = pd.DataFrame(data)

# Compute means and standard deviations
summary_stats = df.groupby("model")["distance_loss"].agg(mean="mean", std="std").reset_index()

# Create the bar chart
plt.figure(figsize=(10, 6))
sns.barplot(data=summary_stats, x="model", y="mean", palette="inferno", ci=None)

# Add custom error bars for standard deviation
plt.errorbar(
    x=range(len(summary_stats)),  # X-coordinates of bars
    y=summary_stats["mean"],  # Mean values
    yerr=summary_stats["std"],  # Standard deviations
    fmt="none",  # No markers for the error bars
    c="black",  # Color of error bars
    capsize=5,  # Length of the caps on the error bars
)

# Overlay individual data points for clarity
sns.stripplot(data=df, x="model", y="distance_loss", color="black", size=7, jitter=True, alpha=0.2)


# Add labels and title
plt.xlabel("Machine Learning Model", fontsize=12)
plt.ylabel("Geographic Distance Loss (km)", fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
#TODO figure out which unique IDs are the ones that are far outliers and consider dropping, otherwise will need to rescale to include the deep learning models

In [ ]:
df_LR = pd.DataFrame(df.distance_loss[df.model == 'Linear Regression'])
df_SVR = pd.DataFrame(df.distance_loss[df.model == 'SVR'])
df_DT = pd.DataFrame(df.distance_loss[df.model == 'Decision Tree'])
df_KNN = pd.DataFrame(df.distance_loss[df.model == 'k-NN'])
df_XG = pd.DataFrame(df.distance_loss[df.model == 'XGBoost'])
df_RF = pd.DataFrame(df.distance_loss[df.model == 'Random Forest'])



df_models = [("Linear Regression", df_LR),
             ("SVR", df_SVR),
             ("Decision Tree", df_DT),
             ("k-NN", df_KNN),
             ("XGBoost", df_XG),
             ("Random Forest", df_RF)]

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from geopy.distance import geodesic

# Make panel fig
method_dict = {
    'Linear Regression': preds_LR, 
    'Support Vector Regressor': preds_SVR,  
    'Decision Tree': preds_DTR, 
    'k-Nearest Neighbors': preds_KNR, 
    'XGBoost': preds_XG_best,
    'Random Forest': preds_RF
}

scaler = sc_y

# Compute global min/max distance across all models
all_distances = []
for preds in method_dict.values():
    unscaled_y_test = scaler.inverse_transform(y_test)
    unscaled_preds = scaler.inverse_transform(preds)
    
    lat_test, lon_test = unscaled_y_test[:, 0], unscaled_y_test[:, 1]
    lat_pred, lon_pred = unscaled_preds[:, 0], unscaled_preds[:, 1]

    distances = np.array([ 
        geodesic((lat1, lon1), (lat2, lon2)).kilometers 
        for lat1, lon1, lat2, lon2 in zip(lat_test, lon_test, lat_pred, lon_pred)
    ])
    
    all_distances.extend(distances)

global_vmin = np.min(all_distances)
global_vmax = np.max(all_distances)  # Ensure this captures the actual max (1275 km)

# Create a shared normalization instance
norm = mcolors.Normalize(vmin=global_vmin, vmax=global_vmax)

# Force df_models to use the correct scaling
df_models_scaled = [(name, (df - df.min().min()) / (df.max().max() - df.min().min()) * (global_vmax - global_vmin) + global_vmin) 
                    for name, df in df_models]

# Create the figure and axes
fig, axes = plt.subplots(len(df_models), 1, figsize=(12, 15), sharex=True, constrained_layout=True)

# Create a shared colorbar axis
cbar_ax = fig.add_axes([1.01, 0.1, 0.02, 0.8])  # Adjust position and size of the colorbar

# Loop through models and plot each heatmap
for ax, (model_name, data) in zip(axes, df_models_scaled):  # Use scaled data
    sns.heatmap(data.T, ax=ax, cmap="magma", cbar=False, vmin=global_vmin, vmax=global_vmax, 
                xticklabels=False, yticklabels=False)  # Ensure heatmaps use correct vmin/vmax
    ax.set_ylabel(model_name, fontsize=16)

# Add the shared colorbar (fix normalization)
sm = plt.cm.ScalarMappable(cmap="magma", norm=norm)  # Use global normalization
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='vertical')
cbar.set_label('Distance Loss (km)', fontsize=16)

# Add a common X-axis label
axes[-1].set_xlabel("Individual Test Sample", fontsize=16)

# Save the plot as a PDF (make sure to do this after plt.show() or tight_layout)
plt.savefig("../pollenGeolocation_saved/figs/sample_compare_all_mods.pdf", format="pdf", bbox_inches="tight")

plt.show()


In [ ]:
""" # plot real vs predicted scatterplots for each model 

method_dict = {'Linear Regression':preds_LR, 
          'SVR':preds_SVR ,  
          'Decision Tree':preds_DTR, 
          'k-NN':preds_KNR, 
          'XGBoost':preds_XG_best,
          'Random Forest':preds_RF}

scaler = sc_y

# Create a 3x2 grid of subplots
fig, axes = plt.subplots(3, 2, figsize=(12, 15))  # 3 rows, 2 columns
axes = axes.flatten()  # Flatten the 2D array for easy iteration

# Iterate over method_dict to access both model_type and associated preds
for i, (model_type, preds) in enumerate(method_dict.items()):
    ax = axes[i]  # Get the subplot axis
    plot_test_preds(y_test, preds, sc_y, model_type, ax=ax)  # Call the plot function

    # Add panel label in the upper left corner
    ax.text(
        -0.15, 1.1,  # Coordinates in axis-relative units (-0.1 from left, 1.1 above top)
        f"{ascii_uppercase[i]}.",  # Label: A., B., C., ...
        transform=ax.transAxes,  # Use axis-relative coordinates
        fontsize=20,  # Font size
        va='top',  # Vertical alignment
        ha='left'  # Horizontal alignment
    )


# Adjust layout for better spacing
plt.tight_layout()

# Save the plot as a PDF (make sure to do this after plt.show() or tight_layout)
plt.savefig("../pollenGeolocation_saved/figs/real_vs_pred_scatterplots.pdf", format="pdf")

plt.show() """

In [ ]:


# Create a 3x2 grid of subplots with adjusted spacing
fig, axes = plt.subplots(3, 2, figsize=(15, 15))  # Increased width for better spacing
axes = axes.flatten()  # Flatten the 2D array for easy iteration

sc_list = []  # Store scatter objects for colorbar reference

# Iterate over method_dict to access both model_type and associated preds
for i, (model_type, preds) in enumerate(method_dict.items()):
    ax = axes[i]  # Get the subplot axis
    sc = plot_test_preds(y_test, preds, sc_y, model_type, ax=ax, norm=norm)  # Pass shared norm
    sc_list.append(sc)

    # Adjust panel label to move it up and left slightly
    ax.text(
        -0.15, 1.15,  # Adjusted coordinates
        f"{ascii_uppercase[i]}.",  # Label: A., B., C., ...
        transform=ax.transAxes,  
        fontsize=20,  
        va='top',  
        ha='right'
    )

# Adjust subplot spacing to create a small gap between columns
plt.subplots_adjust(wspace=0.3, hspace=0.4)  # `wspace` increases horizontal gap

# Add a single horizontal colorbar across the bottom
cbar_ax = fig.add_axes([0.2, 0.05, 0.6, 0.02])  # [left, bottom, width, height]
cbar = plt.colorbar(sc_list[-1], cax=cbar_ax, orientation="horizontal")
cbar.set_label('Prediction Error (km)', fontsize=12)

# Save the plot as a PDF
plt.savefig("../pollenGeolocation_saved/figs/real_vs_pred_scatterplots_normalized.pdf", format="pdf")

plt.show()


In [ ]:
# TODO plot the distance between real and predicted values for all models together 
# envisioning a boxplot or violin plot with distance bt real and preds as the y and on the x would be
# each model with its own violin

In [ ]:
# TODO add random forest map plot with arrows connecting predicted to real 